In [0]:
# ======================================
# Dataset: olist_customers
# Layer: Silver
# Source: Bronze (Delta)
# Grain: 1 row per customer_id
# ======================================

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
%run ../commons/silver_utils

In [0]:
%run ../../00_config/paths

In [0]:
df_bronze = spark.read.format('delta').load(BRONZE_CUSTOMERS_PATH)
df_bronze.printSchema()

In [0]:
df = normalize_columns(df_bronze)

In [0]:
df = df.select(
    F.col("customer_id"),
    F.col("customer_unique_id"),
    F.col("customer_zip_code_prefix"),
    F.trim(F.lower(F.col("customer_city"))).alias("customer_city"),
    F.upper(F.col("customer_state")).alias("customer_state")
)

In [0]:
df = df.filter(
    F.col("customer_id").isNotNull() &
    F.col("customer_unique_id").isNotNull()
)


In [0]:
df = df.filter(F.length(F.col("customer_state")) == 2)


In [0]:
df = df.filter(F.length(F.col("customer_zip_code_prefix")) >= 4)

In [0]:
df.printSchema()

In [0]:
print("Bronze count:", df_bronze.count())
print("Silver count:", df.count())


In [0]:
write_silver(df, SILVER_CUSTOMERS_PATH)